# ทำนายประเภทหนังสือ: ลบ Missing Value และแบ่ง Train/Test (80/20)
ไฟล์ข้อมูล: `books_data.csv` (วางไว้โฟลเดอร์เดียวกับ notebook นี้)

## 1. โหลดข้อมูลและดู missing value

In [2]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split

df = pd.read_csv("bataset/books_data.csv")
print("ขนาดข้อมูลเริ่มต้น:", df.shape)

missing = df.isna().sum().to_frame("จำนวนที่หาย")
missing["เปอร์เซ็นต์"] = (missing["จำนวนที่หาย"] / len(df) * 100).round(2)
missing

ขนาดข้อมูลเริ่มต้น: (212404, 10)


,จำนวนที่หาย,เปอร์เซ็นต์
Title,1,0.00
description,68442,32.22
authors,31413,14.79
image,52075,24.52
previewLink,23836,11.22
publisher,75886,35.73
publishedDate,25305,11.91
infoLink,23836,11.22
categories,41199,19.40
ratingsCount,162652,76.58


## 2. เตรียมตัวแปรเป้าหมาย (`categories`) และปีที่พิมพ์
- `categories` เก็บเป็นข้อความรูปแบบ list เช่น `"['Fiction']"` จึงดึงค่าแรกออกมา
- ดึงปีจาก `publishedDate` (ค่าที่เจอมีทั้ง `1996`, `2005-02`, `2005-01-01`)

In [3]:
def get_first_category(s):
    """แปลง "['Fiction']" -> 'Fiction' ถ้าอ่านไม่ได้ให้เป็น NaN"""
    if not isinstance(s, str):
        return np.nan
    try:
        items = ast.literal_eval(s)
        return items[0] if len(items) > 0 else np.nan
    except (ValueError, SyntaxError):
        return np.nan

df["category"] = df["categories"].apply(get_first_category)
df["year"] = pd.to_numeric(df["publishedDate"].str.extract(r"^(\d{4})")[0], errors="coerce")

print("แถวที่ไม่มี category:", df["category"].isna().sum())
print("แถวที่ไม่มีปี      :", df["year"].isna().sum())

แถวที่ไม่มี category: 41199
แถวที่ไม่มีปี      : 25442


## 3. ลบ missing value
ลบแถวที่ไม่มี `category` (ตัวแปรเป้าหมาย) และแถวที่ไม่มีปี รวมถึงปีที่ผิดปกติ

> ปรับช่วงปีได้ที่ `YEAR_MIN` / `YEAR_MAX` (ข้อมูลชุดนี้มีหนังสือครบถึงประมาณปี 2021 ส่วนปี 2022 ขึ้นไปมีน้อยมาก)

In [4]:
YEAR_MIN, YEAR_MAX = 1900, 2022

before = len(df)
df_clean = df.dropna(subset=["category", "year"]).copy()
df_clean = df_clean[df_clean["year"].between(YEAR_MIN, YEAR_MAX)].copy()
df_clean["year"] = df_clean["year"].astype(int)

print(f"ก่อนลบ : {before:,} แถว")
print(f"หลังลบ : {len(df_clean):,} แถว  (ตัดออก {before - len(df_clean):,} แถว)")

ก่อนลบ : 212,404 แถว
หลังลบ : 168,304 แถว  (ตัดออก 44,100 แถว)


## 4. เลือกคอลัมน์ที่ใช้ และจัดการคอลัมน์ที่ยังมี missing
ตัดคอลัมน์ URL ที่ไม่ช่วยทำนายประเภท ส่วนคอลัมน์อื่นเติมค่าอย่างง่ายเพื่อไม่ต้องลบแถวเพิ่ม
(`ratingsCount` หายเกิน 70% ถ้าลบทุกแถวที่หายจะเหลือข้อมูลน้อยมาก)

In [5]:
df_clean = df_clean.drop(columns=["image", "previewLink", "infoLink", "categories", "publishedDate"])

# ratingsCount: NaN น่าจะแปลว่ายังไม่มีคนให้คะแนน
df_clean["has_rating"] = df_clean["ratingsCount"].notna().astype(int)
df_clean["ratingsCount"] = df_clean["ratingsCount"].fillna(0)

# ข้อความ/ชื่อ: เติมค่าว่าง + flag
df_clean["has_description"] = df_clean["description"].notna().astype(int)
df_clean["description"] = df_clean["description"].fillna("")
df_clean["authors"] = df_clean["authors"].fillna("Unknown")
df_clean["publisher"] = df_clean["publisher"].fillna("Unknown")
df_clean["Title"] = df_clean["Title"].fillna("")

print("missing ที่เหลือ:", int(df_clean.isna().sum().sum()))

missing ที่เหลือ: 0


## 5. รวมหมวดหนังสือเป็น Top-N + Other
`categories` มีค่าไม่ซ้ำหลายพันแบบ จึงเก็บเฉพาะหมวดที่พบบ่อย ที่เหลือรวมเป็น `Other`

In [6]:
TOP_N = 15

top_cats = df_clean["category"].value_counts().head(TOP_N).index
df_clean["category_grouped"] = df_clean["category"].where(df_clean["category"].isin(top_cats), "Other")

df_clean["category_grouped"].value_counts()

category_grouped
Other                        81922
Fiction                      23335
Religion                      9402
History                       9309
Juvenile Fiction              6630
Biography & Autobiography     6309
Business & Economics          5620
Computers                     4308
Social Science                3825
Juvenile Nonfiction           3440
Science                       2613
Education                     2585
Cooking                       2436
Sports & Recreation           2261
Family & Relationships        2172
Literary Criticism            2137
Name: count, dtype: int64

## 6. แบ่ง Train 80% / Test 20%
ใช้ `stratify` เพื่อให้สัดส่วนของแต่ละหมวดใน train และ test ใกล้เคียงกัน

In [7]:
X = df_clean.drop(columns=["category", "category_grouped"])
y = df_clean["category_grouped"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    train_size=0.80,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print(f"สัดส่วน train = {len(X_train)/len(X):.0%}, test = {len(X_test)/len(X):.0%}")

Train: (134643, 8) | Test: (33661, 8)
สัดส่วน train = 80%, test = 20%


ตรวจว่าสัดส่วนแต่ละหมวดใน train/test ใกล้เคียงกัน

In [8]:
compare = pd.DataFrame({
    "train_%": (y_train.value_counts(normalize=True) * 100).round(2),
    "test_%":  (y_test.value_counts(normalize=True) * 100).round(2),
})
compare

,train_%,test_%
category_grouped,,
Other,48.67,48.68
Fiction,13.86,13.86
Religion,5.59,5.59
History,5.53,5.53
Juvenile Fiction,3.94,3.94
Biography & Autobiography,3.75,3.75
Business & Economics,3.34,3.34
Computers,2.56,2.56
Social Science,2.27,2.27


## (ทางเลือก) แบ่งตามเวลา
เพราะโจทย์คือทำนายอนาคต การแบ่งตามปีจะจำลองสถานการณ์จริงได้ดีกว่า: train = ปีเก่า, test = ปีล่าสุด ถ้าจะใช้ ให้เปลี่ยน `USE_TIME_SPLIT = True`

In [9]:
USE_TIME_SPLIT = False

if USE_TIME_SPLIT:
    df_sorted = df_clean.sort_values("year").reset_index(drop=True)
    cut = int(len(df_sorted) * 0.80)
    train_df, test_df = df_sorted.iloc[:cut], df_sorted.iloc[cut:]

    X_train = train_df.drop(columns=["category", "category_grouped"])
    y_train = train_df["category_grouped"]
    X_test = test_df.drop(columns=["category", "category_grouped"])
    y_test = test_df["category_grouped"]

    print("Train ปี:", X_train["year"].min(), "-", X_train["year"].max())
    print("Test  ปี:", X_test["year"].min(), "-", X_test["year"].max())

## 7. บันทึกเป็นไฟล์ train / test
บันทึกชุดที่แบ่งไว้ (ถ้าเปิด `USE_TIME_SPLIT` ก็จะบันทึกชุดที่แบ่งตามเวลา) รวมคอลัมน์เป้าหมาย `category_grouped` ไว้ในไฟล์เดียวกัน

In [ ]:
train_out = X_train.copy()
train_out["category_grouped"] = y_train
test_out = X_test.copy()
test_out["category_grouped"] = y_test

train_out.to_csv("train.csv", index=False, encoding="utf-8")
test_out.to_csv("test.csv", index=False, encoding="utf-8")

print("บันทึกแล้ว: train.csv", train_out.shape, "| test.csv", test_out.shape)   

บันทึกแล้ว: train.csv (134643, 9) | test.csv (33661, 9)
